[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day9_lab.ipynb)

# Day 9 · 실습 — 에이전트의 구조

설비 일지를 대신 찾아 주는 비서를 만든다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

실습 시간에 푼다. **강의 노트북에 없던 문제**들이다.

`스스로 풀기` 는 각자, `조별로 풀기` 는 2~3명이 한 조로 상의하며 푼다.
막히면 강의 노트북(`live`)에서 같은 함수를 쓴 셀을 찾아 대조한다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 준비

In [ ]:
# 키는 화면에 안 찍히게 받는다. 붙여 넣고 Enter 를 누르면 된다.
import getpass, json, urllib.request
KEY = getpass.getpass('nvapi- 로 시작하는 키: ')
print('키 길이', len(KEY))     # 60~80 정도면 제대로 들어간 것이다

In [ ]:
# 모델에 대화를 통째로 보내는 함수. 실패해도 노트북이 멈추지 않게 [실패] 를 돌려준다.
URL = 'https://integrate.api.nvidia.com/v1/chat/completions'
MODEL = 'nvidia/llama-3.3-nemotron-super-49b-v1'

def chat(messages, tools=None, n=400, temp=0):
    body = {'model': MODEL, 'max_tokens': n, 'temperature': temp,
            'messages': messages}
    if tools:                       # 도구 목록은 있을 때만 같이 보낸다
        body['tools'] = tools
    req = urllib.request.Request(URL, data=json.dumps(body).encode(), headers={
        'Authorization': 'Bearer ' + KEY,
        'Content-Type': 'application/json', 'Accept': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=120) as f:
            return json.load(f)['choices'][0]['message']
    except Exception as e:
        return {'role': 'assistant', 'content': '[실패] %s' % str(e)[:80]}

In [ ]:
# 한 문장만 물어볼 때 쓰는 짧은 이름
def say(text, n=200):
    return (chat([{'role': 'user', 'content': text}], n=n).get('content') or '').strip()

print(say('한 단어로만 답하라. 대한민국의 수도는?', 10))   # '서울' 이 나오면 준비 끝이다

## 4. 도구를 만들어 준다

In [ ]:
# 어제 하루치 설비 일지 — 현업에서는 이 자리가 DB 조회나 엑셀 읽기가 된다
LOG = {
    '1호기': {'라인': 'A', '생산': 1240, '불량': 30, '근무조': '주간'},
    '2호기': {'라인': 'A', '생산':  980, '불량': 11, '근무조': '야간'},
    '3호기': {'라인': 'B', '생산': 1530, '불량': 28, '근무조': '주간'},
    '4호기': {'라인': 'B', '생산':  760, '불량': 24, '근무조': '야간'},
}
for k, v in LOG.items():
    print('%s  %s라인  생산 %5d  불량 %3d' % (k, v['라인'], v['생산'], v['불량']))

In [ ]:
# 함수 셋 — 계산기 · 오늘 날짜 · 설비 조회
import datetime

def calc(expr):
    """산술식 하나를 계산해 문자열로 돌려준다"""
    return str(eval(expr, {'__builtins__': {}}, {}))

def today():
    """오늘 날짜"""
    return datetime.date.today().isoformat()

def machine_info(machine):
    """설비 한 대의 어제 기록. 없는 이름이면 쓸 수 있는 이름을 알려 준다."""
    v = LOG.get(machine)
    if v is None:
        return '그런 설비는 없다. 쓸 수 있는 이름: ' + ', '.join(LOG)
    return '%s: %s라인, 생산 %d개, 불량 %d개, 근무조 %s' % (
        machine, v['라인'], v['생산'], v['불량'], v['근무조'])

FUNCS = {'calc': calc, 'today': today, 'machine_info': machine_info}
print(calc('17*24'))
print(today())
print(machine_info('3호기'))
print(machine_info('9호기'))     # 없는 이름을 넣으면 이렇게 알려 준다

In [ ]:
# 모델에게 건네는 도구 목록. spec() 은 매번 같은 모양을 찍어 주는 짧은 도우미다.
def spec(name, desc, props, required):
    return {'type': 'function', 'function': {
        'name': name, 'description': desc,
        'parameters': {'type': 'object', 'properties': props, 'required': required}}}

TOOLS = [
    spec('calc', '산술식을 계산한다. 나눗셈·퍼센트처럼 정확한 값이 필요할 때 쓴다.',
         {'expr': {'type': 'string', 'description': '파이썬 산술식. 예: 28/1530*100'}}, ['expr']),
    spec('today', '오늘 날짜를 YYYY-MM-DD 로 돌려준다.', {}, []),
    spec('machine_info', '설비 한 대의 어제 생산량·불량 수·라인·근무조를 사내 일지에서 찾아 돌려준다.',
         {'machine': {'type': 'string', 'description': '설비 이름. 예: 3호기'}}, ['machine']),
]
print(json.dumps(TOOLS[2], ensure_ascii=False, indent=1))

## 6. 루프 — 판단 · 행동 · 관찰

In [ ]:
# 에이전트 본체. 마지막 대화는 LAST 에 남겨 두었다가 9절에서 다시 본다.
SYSTEM = '너는 공정 데이터 비서다. 필요하면 도구를 부르고, 모르면 모른다고 답한다.'
LAST = []

def run_agent(question, system=SYSTEM, max_steps=5, log=True):
    global LAST
    messages = [{'role': 'system', 'content': system},
                {'role': 'user', 'content': question}]
    for step in range(max_steps):
        m = chat(messages, TOOLS, 500)           # ① 판단 — 부를까, 답할까
        messages.append(m)
        calls = m.get('tool_calls') or []
        if not calls:                            # 부를 것이 없으면 그것이 답이다
            LAST = messages
            return m.get('content') or ''
        for c in calls:                          # ② 행동 — 고른 도구를 실행
            name = c['function']['name']
            args = json.loads(c['function']['arguments'] or '{}')
            out = FUNCS[name](**args)
            if log:
                print('  [도구] %s(%s) -> %s' % (name, args, out))
            messages.append({'role': 'tool', 'tool_call_id': c['id'],
                             'content': out})    # ③ 관찰 — 결과를 대화에 되먹인다
    LAST = messages
    return '[한도] %d번 안에 못 끝냈다' % max_steps

## 8. 안전장치

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 1.** 위 질문이 **끝까지 가는 최소 한도**를 찾는다.
> `2` 부터 하나씩 올려 가며 `[한도]` 가 안 뜨는 값을 찾는다.

In [ ]:
# 한도는 작으면 답이 안 나오고, 크면 비용이 샌다
LIMIT = ___

print(run_agent('1호기부터 4호기까지 평균 불량률을 알려줘', max_steps=LIMIT))

## 9. 컨텍스트 — 대화가 얼마나 커지는가

In [ ]:
# 대화를 글로 펼쳐 요약을 받는다
def flatten(messages):
    return '\n'.join('%s: %s' % (m['role'], str(m.get('content'))[:200])
                     for m in messages)

## 10. 내 업무로

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 1.** `SYSTEM` 한 줄만 바꿔 답의 모양을 고정한다.
> 「너는 무엇인가 · 언제 도구를 부르나 · 어떤 형식으로 답하나」 세 가지를 적는다.
> 같은 질문에 답이 어떻게 달라지는지 앞 절과 견줘 본다.

In [ ]:
MY_SYSTEM = '너는 ___ 다. ___ 할 때만 도구를 부르고, ___ 형식으로만 답한다.'
print(run_agent('3호기 어제 불량률 알려줘', system=MY_SYSTEM))
print('같은 도구라도 시스템 프롬프트가 행동을 바꾼다')

> **실습문제 2.** `LOG` 를 **자기 업무 데이터**로 바꾸고, 도구 설명도 그 말로 고친다.
> 설비가 아니라 거래처 · 품목 · 창고 무엇이든 된다. 열 이름만 자기 말로 바꾸면 된다.
> 설명을 애매하게 적으면 도구를 안 부르거나 엉뚱하게 부른다. 일부러 애매하게도 해 본다.

In [ ]:
LOG = {
    '___': {'___': ___, '___': ___},
    '___': {'___': ___, '___': ___},
}
TOOLS[2]['function']['description'] = '___'
print(run_agent('___ 알려줘'))
print('도구 설명은 모델용 프롬프트다 — 구체적일수록 잘 고른다')

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 3.** 도구를 **하나 더** 붙인다. 함수 하나와 설명 하나면 루프는 그대로 돈다.
> ① 파이썬 함수를 쓴다 → ② `FUNCS` 에 이름을 등록한다 → ③ `TOOLS` 에 설명을 넣는다.
> 세 줄이 끝이다. `run_agent` 는 한 글자도 안 고친다.

In [ ]:
def my_tool(___):
    return '___'

FUNCS['___'] = my_tool
TOOLS.append(spec('___', '___', {'___': {'type': 'string', 'description': '___'}}, ['___']))
print(run_agent('___'))
print('루프는 한 줄도 안 고쳤다 — 도구 목록만 늘었다')

## 11. 반출본 만들기 — 컬럼 이름부터

In [ ]:
# 셀 공정 기록 2,400행을 읽어 온다. 난수로 만든 가상 데이터라 이 파일 자체는 반출 걱정이 없다.
import pandas as pd, numpy as np

URL = 'https://tunalee.github.io/posco/data/cell_process.csv'
df = pd.read_csv(URL, parse_dates=['시각'])
print(df.shape)
print(df.head(3).to_string(index=False))

In [ ]:
# 컬럼마다 (익명명, 변환 방식, 파라미터) 를 정해 둔 표. 이 표가 곧 반출 스크립트 명세다.
#   spec : (x - target) / tol   — 스펙이 있는 값. 관리도·Cpk 가 그대로 나온다
#   z    : (x - 평균) / 표준편차 — 스펙을 모를 때. 상관·회귀가 그대로 나온다
#   rank : 백분위                — 순서만 남긴다. 회귀에는 못 쓴다
RULE = {
    '건조_ZONE2_TEMP':   ('TEMP_D2',  'spec', (120.0, 5.0)),
    '프레스_1호기_압력':  ('PRES_B1',  'z',    None),
    '코팅_로딩_mg_cm2':  ('LOAD_C1',  'z',    None),
    '화성_3단계_CV_전압': ('VOLT_P3',  'z',    None),
    'NMP_투입비':        ('RATIO_M2', 'rank', None),
}
for src, (dst, how, p) in RULE.items():
    print('%-18s -> %-9s %s' % (src, dst, how))

In [ ]:
# 표대로 바꿔 주는 함수. 여기 코드는 안 고친다 — 위의 RULE 만 고친다.
def export(df, rule):
    out = pd.DataFrame(index=df.index)
    for src, (dst, how, p) in rule.items():
        x = df[src].astype(float)
        if   how == 'spec': out[dst] = ((x - p[0]) / p[1]).round(3)
        elif how == 'z':    out[dst] = ((x - x.mean()) / x.std()).round(3)
        elif how == 'rank': out[dst] = x.rank(pct=True).round(3)
    out['LOT'] = pd.factorize(df['로트번호'])[0]        # 순번을 지운다
    out['MC']  = pd.factorize(df['설비호기'])[0]        # 호기 이름을 지운다
    out['T']   = ((df['시각'] - df['시각'].min())
                  .dt.total_seconds() / 3600).round(2)  # t0 기준 시간
    return out

In [ ]:
# 반출본을 만들어 본다
out = export(df, RULE)
print(out.head(3).to_string(index=False))

In [ ]:
# 사람 눈으로 매번 보지 않는다. 검사기를 하나 만들어 두고 그것만 통과시킨다.
def check(out, src, rule):
    bad = []
    ko = [c for c in out.columns if any('가' <= ch <= '힣' for ch in c)]
    if ko:
        bad.append('한글 컬럼명이 남았다: %s' % ko)
    same = [c for c in out.columns if c in src.columns]
    if same:
        bad.append('원본 컬럼명이 그대로다: %s' % same)
    for c in out.columns:
        for s in src.select_dtypes('number').columns:
            if out[c].round(3).equals(src[s].round(3)):
                bad.append('%s 가 원본 %s 와 같은 값이다' % (c, s))
    print('\n'.join(bad) if bad else '내보내도 되는 모양이다')

check(out, df, RULE)

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 4.** `df` 를 **자기 업무 표**로 바꾸고, 컬럼마다 `RULE` 을 채운다.
> ① 익명명은 「물리량_순번」 으로 짓는다 — 공정 이름·설비 이름은 넣지 않는다.
> ② 스펙이 있으면 `spec`, 없으면 `z`, 순서만 보면 되면 `rank`.
> ③ `check()` 가 「내보내도 되는 모양이다」를 찍을 때까지 고친다.

In [ ]:
MY_RULE = {
    '___': ('___', '___', ___),
    '___': ('___', '___', ___),
}
my_out = export(df, MY_RULE)
check(my_out, df, MY_RULE)
print(my_out.head(3).to_string(index=False))